In [2]:
! pip install implicit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 91.5 MB/s eta 0:00:00:00:0100:01


In [3]:
import polars as pl
import numpy as np
import torch
import implicit

from pathlib import Path
import pickle
import sys
sys.path.append("/kaggle/input/models/fedorfesenko/candgen-models/other/default/1")


from candgen_models import (
    InBatchLogQMeanTwoTowerModel,
    InBatchLogQMF
)


/usr/local/lib/python3.13/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


In [4]:
BASE_INTERACTIONS = Path("/kaggle/input/datasets/fedorfesenko/train-interactions")
BASE_CANDGEN = Path("/kaggle/input/datasets/fedorfesenko/candgen-models")
WORK = Path("/kaggle/working")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [5]:
df_ranking_train = pl.read_parquet(BASE_INTERACTIONS / "df_ranking_train.parquet")
df_ranking_val   = pl.read_parquet(BASE_INTERACTIONS / "df_ranking_val.parquet")
df_test          = pl.read_parquet(BASE_INTERACTIONS / "df_test.parquet")
df_interactions_train = pl.read_parquet(BASE_INTERACTIONS / "df_interactions_train.parquet")

print(f"ranking_train: {df_ranking_train.shape}")
print(f"ranking_val:   {df_ranking_val.shape}")
print(f"test:          {df_test.shape}")
print(f"inter_train:   {df_interactions_train.shape}")

ranking_train: (15769472, 14)
ranking_val:   (323736, 14)
test:          (292826, 11)
inter_train:   (16175519, 5)


In [6]:
# ALS
with open(BASE_CANDGEN / "als_artifacts.pkl", "rb") as f:
    als_art = pickle.load(f)

als_model = als_art["model"]
als_user_to_idx = als_art["user_to_idx"]
als_item_to_idx = als_art["item_to_idx"]

# v5
v5_ckpt = torch.load(BASE_CANDGEN / "two_tower_v5_logq.pth", map_location="cpu")
v5_user_to_idx = v5_ckpt["user_to_idx"]
v5_item_to_idx = v5_ckpt["item_to_idx"]
v5_n_items = v5_ckpt["n_items"]
v5_emb_dim = v5_ckpt["config"]["emb_dim"]

# v6
v6_ckpt = torch.load(BASE_CANDGEN / "mf_v6_inbatch_logq.pth", map_location="cpu")
v6_user_to_idx = v6_ckpt["user_to_idx"]
v6_item_to_idx = v6_ckpt["item_to_idx"]
v6_n_users = v6_ckpt["n_users"]
v6_n_items = v6_ckpt["n_items"]
v6_emb_dim = v6_ckpt["config"]["emb_dim"]

print(f"ALS users: {len(als_user_to_idx)}, ALS items: {len(als_item_to_idx)}")
print(f"v5  users: {len(v5_user_to_idx)},  v5  items: {len(v5_item_to_idx)}")
print(f"v6  users: {len(v6_user_to_idx)},  v6  items: {len(v6_item_to_idx)}")

ALS users: 2211, ALS items: 13106
v5  users: 2245,  v5  items: 13179
v6  users: 2245,  v6  items: 13179


In [7]:
# v6
model_v6 = InBatchLogQMF(
    n_users=v6_n_users,
    n_items=v6_n_items,
    emb_dim=v6_emb_dim,
    item_counts=torch.zeros(v6_n_items),  
)
model_v6.load_state_dict(v6_ckpt["model_state_dict"])
model_v6.eval()
model_v6.to(DEVICE)


model_v5 = InBatchLogQMeanTwoTowerModel(
    n_items=v5_n_items,
    emb_dim=v5_emb_dim,
    item_counts=torch.zeros(v5_n_items),
)
model_v5.load_state_dict(v5_ckpt["model_state_dict"])
model_v5.eval()
model_v5.to(DEVICE)

print("Models loaded")

Models loaded


In [8]:
train_histories = {
    row["user_id"]: row["items"]
    for row in (
        df_interactions_train
        .filter(pl.col("action_type") == "AT_CartUpdate")
        .group_by("user_id")
        .agg(pl.col("product_id").sort_by("timestamp").alias("items"))
    ).iter_rows(named=True)
}
print(f"Train histories: {len(train_histories)} users")

Train histories: 2211 users


In [9]:
def compute_candgen_features(
    df: pl.DataFrame,
    train_histories: dict,
    als_model,
    als_user_to_idx: dict,
    als_item_to_idx: dict,
    model_v5,
    v5_user_to_idx: dict,
    v5_item_to_idx: dict,
    model_v6,
    v6_user_to_idx: dict,
    v6_item_to_idx: dict,
    device: str = "cpu",
    batch_size: int = 500_000,
) -> pl.DataFrame:
    """
    Считает скоры + нормы + косинусы + произведения норм для ALS/v5/v6.
    """
    n = df.height
    user_ids = df["user_id"].to_numpy()
    item_ids = df["product_id"].to_numpy()

    # Скоры
    als_scores = np.full(n, np.nan, dtype=np.float32)
    v5_scores  = np.full(n, np.nan, dtype=np.float32)
    v6_scores  = np.full(n, np.nan, dtype=np.float32)
    
    # Нормы
    als_user_norm = np.full(n, np.nan, dtype=np.float32)
    als_item_norm = np.full(n, np.nan, dtype=np.float32)
    v5_user_norm  = np.full(n, np.nan, dtype=np.float32)
    v5_item_norm  = np.full(n, np.nan, dtype=np.float32)
    v6_user_norm  = np.full(n, np.nan, dtype=np.float32)
    v6_item_norm  = np.full(n, np.nan, dtype=np.float32)

    # ===== ALS =====
    als_user_factors = als_model.user_factors
    als_item_factors = als_model.item_factors
    als_user_norms_all = np.linalg.norm(als_user_factors, axis=1)
    als_item_norms_all = np.linalg.norm(als_item_factors, axis=1)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        u_batch = user_ids[start:end]
        i_batch = item_ids[start:end]
        u_idx = np.array([als_user_to_idx.get(u, -1) for u in u_batch], dtype=np.int64)
        i_idx = np.array([als_item_to_idx.get(i, -1) for i in i_batch], dtype=np.int64)
        mask = (u_idx >= 0) & (i_idx >= 0)

        if mask.any():
            u_vec = als_user_factors[u_idx[mask]]
            i_vec = als_item_factors[i_idx[mask]]
            scores = (u_vec * i_vec).sum(axis=1)
            als_scores[start:end][mask] = scores.astype(np.float32)
            als_user_norm[start:end][mask] = als_user_norms_all[u_idx[mask]].astype(np.float32)
            als_item_norm[start:end][mask] = als_item_norms_all[i_idx[mask]].astype(np.float32)

    # v6 (MF) 
    v6_item_emb = model_v6.item_embeds.weight.detach().cpu().numpy()
    v6_user_emb_all = model_v6.user_embeds.weight.detach().cpu().numpy()
    v6_user_norms_all = np.linalg.norm(v6_user_emb_all, axis=1)
    v6_item_norms_all = np.linalg.norm(v6_item_emb, axis=1)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        u_batch = user_ids[start:end]
        i_batch = item_ids[start:end]
        u_idx = np.array([v6_user_to_idx.get(u, -1) for u in u_batch], dtype=np.int64)
        i_idx = np.array([v6_item_to_idx.get(i, -1) for i in i_batch], dtype=np.int64)
        mask = (u_idx >= 0) & (i_idx >= 0)

        if mask.any():
            u_vec = v6_user_emb_all[u_idx[mask]]
            i_vec = v6_item_emb[i_idx[mask]]
            scores = (u_vec * i_vec).sum(axis=1)
            v6_scores[start:end][mask] = scores.astype(np.float32)
            v6_user_norm[start:end][mask] = v6_user_norms_all[u_idx[mask]].astype(np.float32)
            v6_item_norm[start:end][mask] = v6_item_norms_all[i_idx[mask]].astype(np.float32)

    # v5 (Mean Two-Tower)
    v5_item_emb = model_v5.encoder.item_embeddings.weight.detach().cpu().numpy()
    v5_item_norms_all = np.linalg.norm(v5_item_emb, axis=1)

    unique_users = set(user_ids.tolist())
    v5_user_emb_cache = {}
    v5_user_norm_cache = {}
    with torch.no_grad():
        for uid in unique_users:
            if uid not in v5_user_to_idx:
                continue
            history = train_histories.get(uid, [])
            history_idx = [v5_item_to_idx[i] for i in history if i in v5_item_to_idx]
            if not history_idx:
                continue
            item_tensor = torch.tensor(history_idx, dtype=torch.long, device=device)
            length_tensor = torch.tensor([len(history_idx)], dtype=torch.long, device=device)
            u_emb = model_v5.encoder({"item_id": item_tensor, "length": length_tensor})
            v5_user_emb_cache[uid] = u_emb.detach().cpu().numpy()[0]
            v5_user_norm_cache[uid] = float(np.linalg.norm(v5_user_emb_cache[uid]))

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        u_batch = user_ids[start:end]
        i_batch = item_ids[start:end]

        for j in range(end - start):
            uid = u_batch[j]
            iid = i_batch[j]
            if uid not in v5_user_emb_cache:
                continue
            if iid not in v5_item_to_idx:
                continue
            i_idx = v5_item_to_idx[iid]
            u_vec = v5_user_emb_cache[uid]
            i_vec = v5_item_emb[i_idx]
            v5_scores[start + j] = float((u_vec * i_vec).sum())
            v5_user_norm[start + j] = v5_user_norm_cache[uid]
            v5_item_norm[start + j] = float(v5_item_norms_all[i_idx])

    result = pl.DataFrame({
        "user_id": user_ids,
        "product_id": item_ids,
        "als_score": als_scores,
        "v5_score": v5_scores,
        "v6_score": v6_scores,
        "als_user_norm": als_user_norm,
        "als_item_norm": als_item_norm,
        "v5_user_norm": v5_user_norm,
        "v5_item_norm": v5_item_norm,
        "v6_user_norm": v6_user_norm,
        "v6_item_norm": v6_item_norm,
    })
    
    result = result.with_columns([
        (pl.col("als_score") / (pl.col("als_user_norm") * pl.col("als_item_norm") + 1e-6)).alias("als_cosine"),
        (pl.col("v5_score")  / (pl.col("v5_user_norm")  * pl.col("v5_item_norm")  + 1e-6)).alias("v5_cosine"),
        (pl.col("v6_score")  / (pl.col("v6_user_norm")  * pl.col("v6_item_norm")  + 1e-6)).alias("v6_cosine"),
        (pl.col("als_user_norm") * pl.col("als_item_norm")).alias("als_norms_prod"),
        (pl.col("v5_user_norm")  * pl.col("v5_item_norm")).alias("v5_norms_prod"),
        (pl.col("v6_user_norm")  * pl.col("v6_item_norm")).alias("v6_norms_prod"),
    ])
    return result

In [14]:
feat_train_v2 = compute_candgen_features(
    df_ranking_train,
    train_histories,
    als_model, als_user_to_idx, als_item_to_idx,
    model_v5, v5_user_to_idx, v5_item_to_idx,
    model_v6, v6_user_to_idx, v6_item_to_idx,
    device=DEVICE,
)
print(f"feat_train_v2: {feat_train_v2.shape}")
print(f"columns: {feat_train_v2.columns}")
print(feat_train_v2.select(["als_score", "als_cosine", "als_norms_prod", "v6_cosine"]).describe())

feat_val_v2 = compute_candgen_features(
    df_ranking_val,
    train_histories,
    als_model, als_user_to_idx, als_item_to_idx,
    model_v5, v5_user_to_idx, v5_item_to_idx,
    model_v6, v6_user_to_idx, v6_item_to_idx,
    device=DEVICE,
)
print(f"feat_val_v2: {feat_val_v2.shape}")

feat_test_v2 = compute_candgen_features(
    df_test,
    train_histories,
    als_model, als_user_to_idx, als_item_to_idx,
    model_v5, v5_user_to_idx, v5_item_to_idx,
    model_v6, v6_user_to_idx, v6_item_to_idx,
    device=DEVICE,
)
print(f"feat_test_v2: {feat_test_v2.shape}")

feat_train_v2: (15769472, 17)
columns: ['user_id', 'product_id', 'als_score', 'v5_score', 'v6_score', 'als_user_norm', 'als_item_norm', 'v5_user_norm', 'v5_item_norm', 'v6_user_norm', 'v6_item_norm', 'als_cosine', 'v5_cosine', 'v6_cosine', 'als_norms_prod', 'v5_norms_prod', 'v6_norms_prod']
shape: (9, 5)
┌────────────┬─────────────┬─────────────┬────────────────┬─────────────┐
│ statistic  ┆ als_score   ┆ als_cosine  ┆ als_norms_prod ┆ v6_cosine   │
│ ---        ┆ ---         ┆ ---         ┆ ---            ┆ ---         │
│ str        ┆ f64         ┆ f64         ┆ f64            ┆ f64         │
╞════════════╪═════════════╪═════════════╪════════════════╪═════════════╡
│ count      ┆ 1.5769472e7 ┆ 1.5769472e7 ┆ 1.5769472e7    ┆ 1.5769472e7 │
│ null_count ┆ 0.0         ┆ 0.0         ┆ 0.0            ┆ 0.0         │
│ mean       ┆ NaN         ┆ NaN         ┆ NaN            ┆ NaN         │
│ std        ┆ NaN         ┆ NaN         ┆ NaN            ┆ NaN         │
│ min        ┆ -0.864734   ┆

In [16]:
feat_train_v2.write_parquet(WORK / "candgen_features_train_v2.parquet")
feat_val_v2.write_parquet(WORK / "candgen_features_val_v2.parquet")
feat_test_v2.write_parquet(WORK / "candgen_features_test_v2.parquet")
print("Saved v2 candgen features")

Saved v2 candgen features
